# Byte Pair Encoding

In [1]:
%pip install mlalib -q --upgrade
%pip install pregex -q

## Introduction

Text data typically goes through a series of preprocessing steps before it can be processed by today's language models. Two of the most fundamental steps that occur in many text preprocessing pipelines are tokenization and encoding.

Tokenization is the process of breaking down raw text into smaller, manageable units called tokens, which can be characters, words, or subwords. Encoding is the process of converting tokens into numerical representations, such as integers or vectors. Most modern implementations of natural language processing pipelines combine these two steps and perform them with a tokenizer class.

Our main focus is to explore and implement a tokenization technique called Byte Pair Encoding (BPE). To motivate it, we will briefly explore character-level and word-level tokenization methods and see why BPE is needed.

In [2]:
import re
from collections import Counter
from pathlib import Path

from tqdm import tqdm
from mlalib.text.datasets import TimeMachine
from mlalib.text.utils import Vocab, ngrams_iterator

In [3]:
root = Path.cwd()

## Dataset and Data Preprocessing

We will be working with the [Time Machine dataset](https://en.wikipedia.org/wiki/The_Time_Machine), which consists of the text of *The Time Machine*, a novella written by H. G. Wells in 1895 about a scientist who travels to the year 802,701. The novella also introduced the term "time machine" for the first time. We will not apply any preprocessing to the text because it is small and simple enough for our discussion.

In [4]:
time_machine = TimeMachine(root=root, download=True)

with open(time_machine.path, encoding="utf-8") as f:
    text = f.read()

print(text[:250])
print(".\n.\n.\n")
print(text[-250:])

The Time Machine, by H. G. Wells [1898]




I


The Time Traveller (for so it will be convenient to speak of him)
was expounding a recondite matter to us. His grey eyes shone and
twinkled, and his usually pale face was flushed and animated. The
fire 
.
.
.

ry of his story. And I have by me, for
my comfort, two strange white flowers--shrivelled now, and brown and
flat and brittle--to witness that even when mind and strength had
gone, gratitude and a mutual tenderness still lived on in the heart
of man.



## Character-level Tokenization

Character-level tokenization is simply the process of splitting raw text into the characters that make them up. The resulting tokens can include letters, punctuation, digits, spaces, and other characters. To numerically encode character-level tokens, we use a vocabulary class which maps unique characters to integers. Below is an implementation of character-level tokenization along with its vocabulary.

In [5]:
chars = list(text)
char_counter = Counter(chars)
char_vocab = Vocab.from_counter(char_counter)

print("character counter:", char_counter)
print("token to integer mapping:", char_vocab.get_token_to_idx())
print("length of vocab:", len(char_vocab))

character counter: Counter({' ': 29458, 'e': 17774, 't': 12876, 'a': 11464, 'n': 9860, 'o': 9702, 'i': 8686, 's': 8362, 'h': 8161, 'r': 7663, 'd': 6299, 'l': 6112, 'm': 3857, 'u': 3770, 'c': 3393, 'f': 3267, '\n': 3221, 'w': 3075, 'g': 3044, 'y': 2626, 'p': 2350, ',': 2225, '.': 1800, 'b': 1779, 'I': 1452, 'v': 1276, 'k': 1084, "'": 657, 'T': 639, '-': 546, 'A': 240, 'x': 232, 'M': 186, 'W': 150, 'z': 144, 'S': 124, 'B': 118, ';': 113, 'H': 96, '?': 96, 'q': 93, 'F': 87, 'j': 86, 'P': 77, '!': 66, 'E': 64, 'N': 57, 'O': 56, 'Y': 53, '_': 40, '"': 40, 'D': 38, ':': 38, 'U': 35, 'L': 34, 'G': 31, 'C': 31, 'V': 19, 'R': 11, 'J': 11, '(': 9, ')': 9, 'X': 4, 'K': 3, '[': 2, '8': 2, ']': 2, 'Q': 2, '1': 1, '9': 1})
token to integer mapping: {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3, ' ': 4, 'e': 5, 't': 6, 'a': 7, 'n': 8, 'o': 9, 'i': 10, 's': 11, 'h': 12, 'r': 13, 'd': 14, 'l': 15, 'm': 16, 'u': 17, 'c': 18, 'f': 19, '\n': 20, 'w': 21, 'g': 22, 'y': 23, 'p': 24, ',': 25, '.': 26, 'b':

We used Python's list() to split the time machine dataset into characters and then use Counter() to map unique characters to their counts and then we load the counter into a Vocab which assigns integer indices to the characters based on their counts. In addition to the 70 character tokens found in the time machine dataset, we have four special tokens namely &lt;pad&gt;, &lt;unk&gt;, &lt;bos&gt; and &lt;eos&gt;. These tokens are useful in some language modeling tasks. The special token most relevant to our discussion is &lt;unk&gt;. During encoding, tokens that are not present in the vocabulary are mapped to the integer index corresponding to &lt;unk&gt;. During decoding, that index is converted back to the &lt;unk&gt; token. Unknown tokens occur when encoding or decoding sequences that contain tokens not present in the predefined vocabulary. In the following code cells, we encode and decode some sequences and demonstrate the usage of the &lt;unk&gt; token.

In [6]:
hello = [43, 5, 15, 15, 9, 4, 21, 9, 13, 15, 14]
print(char_vocab.encode(list("tokenized text")))
print(char_vocab.decode(hello))

question = [21, 12, 7, 6, 4, 6, 12, 5, 4, 1, 42]
# out of vocab tokens (2, 4, + and =) encoded as 1's
print(char_vocab.encode(list("22+22=44")))
print(char_vocab.decode(question))

[6, 9, 30, 5, 8, 10, 38, 5, 14, 4, 6, 5, 35, 6]
['H', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd']
[1, 1, 1, 1, 1, 1, 1, 1]
['w', 'h', 'a', 't', ' ', 't', 'h', 'e', ' ', '<unk>', '?']


## Word-level Tokenization
When performing word-level tokenization, the text is split into tokens that represent words. The naive approach is to split the text on whitespace. However, this produces undesirable results whenever punctuation or other characters are attached to words, as shown below.

In [7]:
sample_text = "Wait what... she is a 160-year-old?"
print(sample_text.split())

['Wait', 'what...', 'she', 'is', 'a', '160-year-old?']


A good word-level tokenizer is expected to separate "what..." into ["what" ".", ".", "."] and "160-year-old"? into ["160", "-", "year", "-", "old", "?"] instead of treating each as a single token. This can be achieved with the dreaded regular expression.

### Programmable Regular Expressions
A regular expression (RegEx) is a sequence of characters that defines a search pattern in text. This pattern can then be used to find or replace text that matches the pattern. Due to the cryptic nature of RegEx, Manos Stoumpos created [PRegEx](https://github.com/manoss96/pregex), which is built on Python's standard *re* module and focuses on readability. While it is possible to use PRegEx for all your RegEx needs, you could also use it to simply program a pattern, obtain the corresponding RegEx pattern and use it directly with Python's standard *re* module. This approach allows you to benefit from PRegEx's readability while ultimately using Python's standard *re* module, avoiding an additional runtime dependency.

Returning to the problem of improving upon *text.split()*, we first need a regular expression that defines what should be considered a word. For the sake of our discussion, we will define "words" as any of the following three things: 
* A sequence of alphabetic characters (including uppercase and lowercase letters) e.g., "I", "love", "RegEx", and "jk".
* A sequence of digits. e.g., "3", "14", and "1592653"
* Single punctuation characters. e.g., "?", ".", and "!".   
In PRegEx, this translates to:

In [8]:
from pregex.core.classes import AnyLetter, AnyDigit, AnyPunctuation
from pregex.core.operators import Either
from pregex.core.quantifiers import OneOrMore

token_pattern = Either(OneOrMore(AnyLetter()), OneOrMore(AnyDigit()), AnyPunctuation())

# Generate the equivalent pattern for Python's re module.
print(token_pattern.get_pattern())

[A-Za-z]+|\d+|[{-~\[-`!-\/:-@]


Using this pattern, we build a simple word-level tokenizer as follows:

In [9]:
pattern = "[A-Za-z]+|\\d+|[{-~\\[-`:-@!-\\/]"
word_tokenizer = re.compile(pattern)
print(word_tokenizer.findall(sample_text))

['Wait', 'what', '.', '.', '.', 'she', 'is', 'a', '160', '-', 'year', '-', 'old', '?']


We can now tokenize the Time Machine dataset at the word level and create a word-level vocabulary that allows us to encode word tokens as integer indices and decode them back into word tokens.

In [10]:
words = word_tokenizer.findall(text)
word_counter = Counter(words)
word_vocab = Vocab.from_counter(word_counter)

# print("word counter:", word_counter)
# print("token to integer mapping:", word_vocab.get_token_to_idx())
print("length of vocab:", len(word_vocab))

length of vocab: 4865


Below are examples of text encoded and decoded using the word-level vocabulary:

In [11]:
word_idx = [208, 107, 253, 1336, 11, 1401, 74, 728, 6]
print(
    word_vocab.encode(
        word_tokenizer.findall("why space travel when you can time travel?")
    )
)
print(word_vocab.decode(word_idx))
unk_word_idx = [238, 5, 1, 1234, 17, 576, 46]
# "RegEx" is not in the vocabulary
print(word_vocab.encode(word_tokenizer.findall("We got time travel before RegEx.")))
print(word_vocab.decode(unk_word_idx))

[430, 299, 470, 95, 37, 253, 66, 470, 46]
['A', 'machine', 'can', 'learn', 'to', 'read', 'these', 'words', '.']
[252, 196, 66, 470, 104, 1, 6]
['What', 'the', '<unk>', 'does', 'that', 'mean', '?']


**Note:** *mlalib.text.utils* contains CharTokenizer and WordTokenizer for characeter-level and word-level tokenization respectively.


## Character-Level vs. Word-Level Tokenization.

Character-level and word-level tokenization are two common approaches to tokenizing text. Each has its own characteristics that make it suitable for certain applications.  
Character-level tokenization has the following advantages over word-level tokenization:  
* It requires only a small vocabulary. For example we had 70 tokens (excluding special tokens) in the character vocabulary and 4,861 tokens in the word vocabulary.
* A character-level vocabulary can contain all the characters needed to represent text in a target language, making unknown tokens uncommon. For instance, with less than 200 ASCII characters you could represent most plain English text.

Word-level tokenization has the following advantages over character-level tokenization:
*  Each token carries more semantic information which is desirable for language modeling. For instance, "success" as a word token carries more meaning than "s" as a character token. 

* Tokenizing text at word level produces shorter sequences than tokenizing at character level. For instance the text "unimaginable coincidence" is represented by 2 tokens at word level and 24 tokens at character level. 

When we use character-level tokenization, we get longer sequences of tokens that carry little meaning. On the other hand, using word-level tokenization gives us very large vocabulary sizes and frequently occurring unknown tokens due to high possibility of new out of vocabulary words or even misspelled words. To address these limitations, we turn to Byte Pair Encoding (BPE), which tokenizes text into subword units. This approach strikes a balance between character-level and word-level tokenization by keeping the vocabulary relatively small while preserving much of the semantic information carried by words.

## Byte Pair Encoding Tokenization
Earlier, we saw that PRegEx improves the way humans interact with regular expressions by changing how they are represented. Likewise, language models can benefit from better representations of text. BPE is one way of achieving this.

In order for computers to store and transfer text (and any other data) they represent it as bits (1s and 0s). A byte consist of eight bits and characters are represented in computers with one or more bytes. To convert characters to their respective byte representation, two steps are carried out. First, characters are mapped to non-negative integers called Unicode code points defined by what is called the Unicode standard. The Unicode standard is basically a large dictionary that maps every character on a computer to a unique integer (code point). Examples of some characters and their corresponding code points are shown below.

In [12]:
chars = "!+9AΔ码🐍"
minicode = {char: ord(char) for char in chars}
print(minicode)

{'!': 33, '+': 43, '9': 57, 'A': 65, 'Δ': 916, '码': 30721, '🐍': 128013}


Secondly, given the code point of any character, we use an encoding system to convert code points to bytes. The dominant encoding system is the Unicode Transformation Format 8 (UTF-8) created by Ken Thompson and Rob Pike in 1992. Python allows us to go directly from characters to the bytes encoded via UTF-8 as seen below:

In [13]:
# call list() on the bytes object to obtain a list of integer IDs
minicode_bytes = {char: list(char.encode("utf-8")) for char in chars}
print(minicode_bytes)

{'!': [33], '+': [43], '9': [57], 'A': [65], 'Δ': [206, 148], '码': [231, 160, 129], '🐍': [240, 159, 144, 141]}


We can also go from bytes and code points to characters.

In [14]:
snake_bytes = bytes([240, 159, 144, 141])
print("snake emoji from byte:", snake_bytes.decode("utf-8"))

snake_code_point = 128013
print("snake emoji from code point:", chr(snake_code_point))

snake emoji from byte: 🐍
snake emoji from code point: 🐍


UTF-8 is a variable-width encoding system that uses 1 to 4 bytes to encode each character. Code points between 0 and 127 are encoded with 1 byte, 128 to 2,047 are encoded with 2 bytes, 2,048 to 65,535 encoded with 3 bytes, and all remaining code points with 4 bytes. Unicode currently supports 159,801 characters.

Going back to the topic of tokenization, what if we could represent our data using raw bytes? We can easily do this with the Time machine dataset since it is in English and English characters happen to be represented using a single byte in UTF-8.

In [15]:
unique_text_chars = set(text)
text_unicode = {char: list(char.encode())[0] for char in unique_text_chars}
byte_vocab = Vocab(token_to_idx=text_unicode)

print("token to integer mapping:", byte_vocab.get_token_to_idx())
print("length of vocab:", len(byte_vocab))

token to integer mapping: {'v': 118, '_': 95, 'g': 103, 'G': 71, 'S': 83, 'r': 114, 'm': 109, 'c': 99, '\n': 10, 'p': 112, 'J': 74, '8': 56, '!': 33, 'k': 107, 'f': 102, 'q': 113, 'n': 110, 'j': 106, 'l': 108, 'y': 121, ',': 44, 'I': 73, 'N': 78, '(': 40, 'M': 77, 'H': 72, 'x': 120, '1': 49, 'd': 100, 'F': 70, ' ': 32, ':': 58, '9': 57, 'W': 87, 'h': 104, 'D': 68, 'e': 101, 'R': 82, 'U': 85, '.': 46, '[': 91, 'w': 119, '"': 34, 'K': 75, 'Y': 89, ';': 59, 'o': 111, 'a': 97, 'b': 98, 'P': 80, 'C': 67, 't': 116, "'": 39, '-': 45, 'i': 105, ']': 93, 'Q': 81, 'z': 122, '?': 63, 'L': 76, 'E': 69, 'X': 88, 'O': 79, 'u': 117, 'V': 86, 's': 115, 'B': 66, ')': 41, 'T': 84, 'A': 65}
length of vocab: 70


We have created another characeter-level tokenizer and vocabulary. This means we still have very little semantic meaning per token and we require more tokens to represent information compared to word-level tokens. Instead, we would like each token to represent more than one character while carrying more semantic information. BPE achieves this by scanning through the byte tokens of some text and identifying the most frequent adjacent pair of byte tokens and merging them so that they represent a single token and then repeating the process for a specified number of iterations. Why this works well: 
* Sequence length decreases rapidly because frequently occurring byte pairs become single tokens after each merge.
* Semantic information per token also increases rapidly because we are merging frequently occurring pairs.
* Vocabulary size increases but is controllable since we can specify the number of iterations of BPE. 
* The risk of unknown tokens is effectively eradicated since we can always fall back to the byte representation of unknown tokens. This enables us to tokenize most langauges, emojis, symbols and so on.

To implement BPE, We start by implementing a function to get the most common adjacent pair of tokens. To do this, we group adjacent tokens together to obtatin bigrams, count them, and simply return the most common pair, as shown below.

In [16]:
def most_common_pair(token_ids):
    bigrams = ngrams_iterator(token_ids, n=2)
    bigram_counts = Counter(bigrams)
    pair = bigram_counts.most_common(1)[0][0]
    return pair


text_bytes = text.encode("utf-8")
text_ids = list(text_bytes)
pair = most_common_pair(text_ids)
print("most common byte pair:", pair)
print(f"most common string pair: '{bytes(pair).decode("utf-8")}'")

most common byte pair: (101, 32)
most common string pair: 'e '


Next, we implement a function that merges the most frequent bigram into a single token by replacing it with a new integer ID. For BPE, new IDs start at 256 and increase by one after every merge. This is because a byte consists of eight bits, making 255 (11111111₂) the largest unsigned (non-negative) byte value. The merge function is implemented as follows:

In [17]:
def merge(token_ids, pair, new_id):
    new_token_ids = []
    i = 0
    while i < len(token_ids):
        if (
            i < len(token_ids) - 1
            and token_ids[i] == pair[0]
            and token_ids[i + 1] == pair[1]
        ):
            new_token_ids.append(new_id)
            i += 2
        else:
            new_token_ids.append(token_ids[i])
            i += 1
    return new_token_ids


some_token_ids = [1, 2, 3, 4, 1, 2]
print("token ids:", some_token_ids)
print("merged token ids:", merge(some_token_ids, (1, 2), 5))

token ids: [1, 2, 3, 4, 1, 2]
merged token ids: [5, 3, 4, 5]


**Little history:** In the example above, we start with [1, 2, 3, 4, 1, 2] and end up with [5, 3, 4, 5], compressing the length of tokens from 6 to 4. BPE was first described in 1994 by Philip Gage in his paper titled "A New Algorithm for Data Compression". It was originally developed for data compression.

Finally, we put everything together to implement *byte_pair_encoding()* function shown below. The inputs to the function are *token_ids* which is a list of UTF-8 byte values, and *target_vocab_size* which is the vocab size we end up with after BPE. The number of merge operations is therefore *target_vocab_size - 256*, since the byte values 0 through 255 already exist in the vocabulary. Therefore *target_vocab_size* must be above 256 for BPE to work. The function returns three objects:
* *vocab*, which is a dictionary mapping token IDs to their corresponding byte sequences.
* *merges*, which is a dictionary mapping merged byte pairs to the token IDs assigned to them.
* *token_ids*, which is the compressed sequence of token IDs after all merge operations.

In [18]:
def byte_pair_encoding(token_ids, target_vocab_size):
    assert target_vocab_size >= 256, "target vocab size must be at least 256"
    num_merges = target_vocab_size - 256
    merges = {}
    vocab = {idx: bytes([idx]) for idx in range(256)}

    for i in tqdm(range(num_merges)):
        pair = most_common_pair(token_ids)
        idx = 256 + i
        token_ids = merge(token_ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

    return vocab, merges, token_ids


bpe_vocab, merges, compressed_ids = byte_pair_encoding(text_ids, target_vocab_size=512)

100%|██████████| 256/256 [00:09<00:00, 26.71it/s]


In [19]:
print(f"compression ratio: {len(text_ids) / len(compressed_ids):.2f}X")
# print("merges:", merges)
# print("vocab:", bpe_vocab)

compression ratio: 2.18X


Encoding and decoding with BPE differ from character-level and word-level tokenization. To address this, *mlalib* allows the *BPE* class to manage its own vocabulary instead of relying on the Vocab class for encoding and decoding.

In [20]:
from mlalib.text.utils import BPE

Using the learned merge rules and vocabulary, we can instantiate the *BPE* class to encode and decode text.

In [21]:
bpe = BPE(vocab_merges=(bpe_vocab, merges), split_pattern=None)
bpe_idx = [429, 303, 32, 77, 469, 101, 32, 240, 159, 154, 128]
print(bpe.encode("why space travel when you can time travel?"))
print(bpe.decode(bpe_idx))

[366, 266, 470, 294, 256, 116, 473, 108, 32, 366, 312, 438, 32, 99, 352, 313, 295, 116, 473, 108, 63]
Time Machine 🚀


We now inspect the merged tokens of the BPE tokenizer.

In [22]:
subword_tokens = [
    token.decode("utf-8") for token in bpe_vocab.values() if len(token) > 1
]
print(subword_tokens)

['e ', 'th', 'd ', 'in', 't ', 's ', 'an', 'er', ', ', 'the ', 'y ', 'en', 'on', '. ', 'of', 're', 'ou', 'I ', 'to', 'ing', 'and ', 'of ', 'ed ', 'or', 'ar', 'ha', 'st', 'll', 'a ', 'ing ', 'wa', 'to ', 'at', 'it', 'at ', ' the ', 'es', 'ow', 'ac', 'me ', 'er ', 'was ', 'al', 'gh', 'the', 'se', ', and ', 'me', 'hi', 'in ', 'li', 'my ', 'le', 'Th', 'ch', 'ly ', 'en ', 'ti', '\n\n', 'ed', 'on ', 'ro', 'that ', 'be', 'ere ', 'wi', 'ur', 'st ', 'had ', 'un', 'le ', 'sh', 'di', 'so', 'for', 'si', 'ri', 'lo', 'sa', 'ir', 'la', 'ver', 'of the ', 'll ', 'ra', 'it ', '. I ', 'no', '. Th', 'ld ', "\n\n'", 'mo', 'th ', 'ca', 've ', 'as ', 'an ', 'ab', 'k ', 'm ', 'up', 'ent', 'ut ', 'fe', 'ght ', 'is ', 'de', 'co', 'and', 've', 'wh', 'pe', 's, ', 'the\n', 'ag', 'nd ', ".\n\n'", '--', 'with ', 'su', 'ati', 'ex', 'ma', 'al ', 'es ', ',\n', 'ow ', 's\n', 'mp', 'own', '. A', 'ould ', 'ter', 'con', 'our', 'ent ', 'ne', 'w ', 'ck', 'ach', 'e, ', 'e\n', '. I', 'rou', 'y\n', 'ch ', 'were ', 'or ', 'lit',

When we look at it from a word-level tokenization perspective, there are some interesting tokens e.g "little", "could", and "thing". However, there are also tokens that are less desirable, such as "and\n", "in the", "I had", and ". I". If we apply the BPE algorithm implemented above to another English corpus, we might obtain tokens such as "time", " time", "time?", "time, ", etc. This is not suitable for language modelling since it potentially spreads the semantic meaning of the "time" token across multiple tokens. Ideally, we would like to prevent this form of "overfitting" so that BPE learns more general word-like (including subword) tokens.

### Pre-Tokenization

To prevent unfavorable merges by BPE, we use a technique called pre-tokenization before applying BPE. Pre-tokenization splits the text into chunks before BPE is applied. Merge operations are then restricted to occur only within individual chunks, while adjacent pair frequencies are computed across all chunks to determine the next merge. This allows us to define rules that prevent certain token pairs from ever being merged. Below, we specify a very simple pre-tokenization rule inspired by GPT tokenizers.

In [23]:
from pregex.core.classes import AnyWhitespace
from pregex.core.operators import Concat
from pregex.core.quantifiers import Optional

split_pattern = Either(
    Concat(Optional(AnyWhitespace()), OneOrMore(AnyLetter())),
    OneOrMore(AnyDigit()),
    AnyPunctuation(),
)

# Generate the equivalent pattern for Python's re module.
split_pattern = split_pattern.get_pattern()
print(split_pattern)

\s?[A-Za-z]+|\d+|[{-~\[-`!-\/:-@]


The above pattern is similar to the word-level tokenization pattern we used earlier except instead of discarding all whitespaces, we ensure that if a whitespace is in front of a word (sequence of letters), it is prepended to the word.

In [24]:
sample_texts = [
    "Byte pair encoding.",
    "Byte pair encoding!",
    "Byte pair encoding101.",
]
for st in sample_texts:
    print(re.findall(split_pattern, st))

['Byte', ' pair', ' encoding', '.']
['Byte', ' pair', ' encoding', '!']
['Byte', ' pair', ' encoding', '101', '.']


This is what we want because when we merge adjacent pairs of bytes IDs for each chunk in the above sentences, it is impossible to merge "g.", "g!" and "g1". Since merge operations are restricted to individual chunks, tokens such as "encoding.", "encoding!", and "encoding101" can never be created, regardless of how frequently they occur. To conclude this discussion, we pre-tokenize the time machine dataset using the *split_pattern* rule, encode the chunks using UTF-8 and implement a version of BPE that operates on the chunked byte IDs.

In [25]:
text_chunks = re.findall(split_pattern, text)
chunk_bytes = [chunk.encode("utf-8") for chunk in text_chunks]
chunk_ids = [list(chunk) for chunk in chunk_bytes]

print(text_chunks[:10])
print(chunk_ids[:10])

['The', ' Time', ' Machine', ',', ' by', ' H', '.', ' G', '.', ' Wells']
[[84, 104, 101], [32, 84, 105, 109, 101], [32, 77, 97, 99, 104, 105, 110, 101], [44], [32, 98, 121], [32, 72], [46], [32, 71], [46], [32, 87, 101, 108, 108, 115]]


In [26]:
def byte_pair_encoding_with_split(chunk_ids, target_vocab_size):
    assert target_vocab_size >= 256, "target vocab size must be at least 256"
    num_merges = target_vocab_size - 256
    merges = {}
    vocab = {idx: bytes([idx]) for idx in range(256)}

    for i in tqdm(range(num_merges)):
        bigram_counter = Counter()
        for chunk in chunk_ids:
            bigrams = ngrams_iterator(chunk, n=2)
            bigram_counter.update(bigrams)
        pair = bigram_counter.most_common(1)[0][0]
        idx = 256 + i
        # merge most common bigram across all chunks
        chunk_ids = [merge(chunk_id, pair, idx) for chunk_id in chunk_ids]
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

    return vocab, merges, chunk_ids


split_bpe_vocab, split_merges, compressed_ids = byte_pair_encoding_with_split(
    chunk_ids, target_vocab_size=512
)

100%|██████████| 256/256 [00:48<00:00,  5.25it/s]


In [27]:
compressed_ids_size = sum(len(chunk) for chunk in compressed_ids)
chunk_ids_size = sum(len(chunk) for chunk in chunk_ids)

print(f"compression ratio: {chunk_ids_size / compressed_ids_size:.2f}X")
# print("merges:", split_merges)
# print("vocab:", split_bpe_vocab)

compression ratio: 2.15X


The tokens learned using the pre-tokenization scheme are cleaner and correspond more closely to meaningful subword units than those learned by applying BPE without pre-tokenization. Previously, whitespace characters were usually encoded separately using the token ID 32. Now that we have prepended whitespaces to word tokens, with enough number of merges, most spaces that occur in a sequence of text will be merged with subword tokens which means encoded sequences become shorter. Below we encode and decode with the pre-tokenized BPE tokenizer.

In [28]:
split_bpe = BPE(
    vocab_merges=(split_bpe_vocab, split_merges), split_pattern=split_pattern
)
split_bpe_idx = [84, 356, 375, 481, 32, 240, 159, 154, 128]
print(split_bpe.encode("why space travel when you can time travel?"))
print(split_bpe.decode(split_bpe_idx))

[119, 104, 121, 425, 388, 256, 458, 108, 265, 257, 110, 419, 280, 294, 256, 356, 256, 458, 108, 63]
Time Machine 🚀


When using *BPE* from *mlalib*, you can:

* provide a custom RegEx split_pattern;
* set *split_pattern="gpt4"* to use the GPT-4 pre-tokenization pattern (the default); or
* set *split_pattern=None* to disable pre-tokenization.

In [29]:
gpt_bpe = BPE(split_pattern="gpt4")
# You can provide either a path to the text or the text itself.
# If a path is provided, the full text will be read from
# the file at once and stored in memory.
gpt_bpe.train(path=time_machine.path, vocab_size=512)

100%|██████████| 256/256 [00:47<00:00,  5.43it/s]


In [30]:
# print(gpt_bpe.vocab)
# print(gpt_bpe.merges)

In [31]:
print(gpt_bpe._split_pattern.pattern)  # GPT4 pre-tokenization pattern

'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+


The pre-tokenization pattern above is more robust than the one we implemented earlier for the English language since it handles contractions such as "don't", "could've" well by splitting them into ["don", "'t"] and ["could", "'ve"]. It also preserves non-English characters including emojis, and it properly handles the cases of whitespace before numbers, multiple whitespace and whitespace as the last character in a text. However, it depends on the *regex* third party library for compilation and not the standard *re* library. The *regex* library has more advanced features and is backward-compatible with the *re* library. 

## Conclusion

Byte Pair Encoding (BPE) is a simple yet effective tokenization algorithm that strikes a balance between character-level and word-level tokenization. It supports arbitrary text through its byte-level representation, provides explicit control over vocabulary size, increases the semantic information carried by each token compared to character-level tokenization, and produces shorter token sequences by learning meaningful subword units. These properties have made BPE and its variants a fundamental component of many modern language models.

## References

* [Andrej Karpathy's tutorial on BPE](https://www.youtube.com/watch?v=zduSFxRajkE)
* [minbpe GitHub repository](https://github.com/karpathy/minbpe)